# Feed Forward Network with PyTorch

## AI Usage

I used AI to investigate and understand errors.

## PyTorch setup

In [123]:
import torch

In [124]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
print(f"Using device: {device}")

Using device: cuda


## Load data and create Datasets

In [125]:
from torchvision.datasets import MNIST
from torchvision import transforms

In [126]:
transform = transforms.ToTensor()
train_data = MNIST(root="data", train=True, download=True, transform=transform)
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [127]:
from torch.utils.data import Subset

Split the training data into training and validation sets

In [128]:
train_indices = list(range(50000))
val_indices = list(range(50000, len(train_data)))

val_data = Subset(train_data, val_indices)
train_data = Subset(train_data, train_indices)
print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")

Training set size: 50000
Validation set size: 10000


In [129]:
test_data = MNIST(root="data", train=False, download=True, transform=transform)
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: ToTensor()

## Defining the Neural Network

In [130]:
import torch.nn as nn

In [131]:
input_size = train_data[0][0].numel()

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

## Defining the optimizer

In [132]:
import torch.optim as optim

In [133]:
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Defining loss function

In [134]:
criterion = nn.CrossEntropyLoss()

## Defining mini-batches creation function

In [135]:
import numpy as np

In [136]:
def create_minibatches(batch_size, x, y):
    total_data = len(x)
    indices = np.arange(total_data)

    for i in range(0, total_data, batch_size):
        batch_idx = indices[i:i + batch_size]
        x_batch = torch.stack([x[j] for j in batch_idx])
        y_batch = torch.tensor([y[j] for j in batch_idx], dtype=torch.long)
        yield x_batch, y_batch

## Train loop

In [137]:
from tqdm import tqdm

In [ ]:
def train(model, train_data, optimizer, criterion, epochs, batch_size):
    # Training data
    x_data = [x for x, _ in train_data]
    y_data = [y for _, y in train_data]

    # Validation data
    x_val = [x for x, y in val_data]
    y_val = [y for x, y in val_data]

    for epoch in range(epochs):
        # Train
        model.train()
        running_loss = 0.0

        epoch_iterator = tqdm(
            create_minibatches(batch_size, x_data, y_data),
            total=(len(train_data) + batch_size - 1) // batch_size,
            desc=f"Epoch {epoch+1}/{epochs}",
            leave=True
        )

        for inputs, labels in epoch_iterator:
            optimizer.zero_grad()
            # The dataset is using cpu, check if we are using gpu and move to gpu
            if inputs.device != device:
                inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(inputs)

        epoch_train_loss = running_loss / len(train_data)

        # Eval
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in create_minibatches(batch_size, [x for x, y in val_data], [y for x, y in val_data]):
                if inputs.device != device:
                    inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * len(inputs)
        
        epoch_val_loss = val_loss / len(val_data)

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}")

    return epoch_train_loss, epoch_val_loss

In [ ]:
loss = train(model, train_data, optimizer, criterion, 30, 64)

Epoch 1/30: 100%|██████████| 782/782 [00:03<00:00, 246.09it/s]


Epoch [1/30], Loss: 0.3979, Val Loss: 0.2047


Epoch 2/30: 100%|██████████| 782/782 [00:03<00:00, 211.10it/s]


Epoch [2/30], Loss: 0.1919, Val Loss: 0.1459


Epoch 3/30: 100%|██████████| 782/782 [00:04<00:00, 189.65it/s]


...

Epoch 29/30: 100%|██████████| 782/782 [00:03<00:00, 208.39it/s]


Epoch [29/30], Loss: 0.0011, Val Loss: 0.1532


Epoch 30/30: 100%|██████████| 782/782 [00:03<00:00, 199.81it/s]


Epoch [30/30], Loss: 0.0108, Val Loss: 0.1202


TypeError: unsupported format string passed to tuple.__format__

In [140]:
print(f"Final Training loss: {loss[0]:.4f}, Final Validation loss: {loss[1]:.4f}")

Final Training loss: 0.0108, Final Validation loss: 0.1202
